# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakshmitejaswiniparuchuri/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

### Lane Choice: Content Refresh (Prioritizing Declining Pages)

I am choosing the **Content Refresh** lane. Across large content portfolios, editorial teams have limited capacity and cannot manually audit or rewrite tens of thousands of existing articles at once. Using historical search performance and page-level metadata from the FlyRank dataset, we can score and rank existing pages so editors focus first on high-visibility articles showing early signs of traffic decay rather than guessing which pages to update.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

### Research Question & Framing

* **Search Question:** Which existing pages with meaningful search visibility are experiencing measurable traffic or CTR decline and should be prioritized first for a content refresh?
* **Unit of Analysis:** One individual **page** (evaluated across historical search performance windows).
* **Output:** A **ranked priority list** (top-K refresh candidates with a priority score and supporting diagnostic signals such as impression trend, CTR gap, and position shift) — not just a raw classification model.
* **Decision & Action:** Each editorial sprint, a content strategist reviews the top-ranked pages to decide **which articles to refresh first**, then assigns an editor to update outdated information, improve headings/meta snippets, or strengthen on-page depth.
* **Cost of a Wrong Recommendation:**
  * **False Positive (flagging a healthy or unrecoverable page):** Editorial hours are wasted rewriting content that did not need changes, and unnecessary edits risk disrupting a stable ranking.
  * **False Negative (missing a genuinely decaying high-value page):** Traffic and clicks continue eroding unnoticed while competitors capture that search demand.
* **Why Data & ML Help (Beyond "Just Train a Model"):** Manually inspecting thousands of URLs across rolling 28-day and 90-day windows is impractical for humans, while naive rules (e.g., "sort by largest click drop") over-index on seasonal topics or tiny low-volume pages. Combining multi-window performance trends, position-adjusted CTR expectations, and page metadata turns noisy search logs into a repeatable editorial triage workflow.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [8]:
import os
import pandas as pd

# Define the GitHub raw URL to download the dataset if it's missing locally
url = "https://raw.githubusercontent.com/lakshmitejaswiniparuchuri/flyrank-ml-internship/main/work/data/raw/content_refresh_anonymized.csv"
local_path = "content_refresh_anonymized.csv"

if not os.path.exists(local_path):
    print("Downloading dataset from repository...")
    try:
        df = pd.read_csv(url)
        df.to_csv(local_path, index=False)
        print("Successfully downloaded and saved dataset locally!")
    except Exception as e:
        print(f"Could not download from main branch: {e}")
        # Fallback to general repository structure try
        url_alt = "https://raw.githubusercontent.com/lakshmitejaswiniparuchuri/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"
        df = pd.read_csv(url_alt)
        df.to_csv(local_path, index=False)
        print("Successfully downloaded from fallback URL!")
else:
    df = pd.read_csv(local_path)
    print("Dataset loaded from local cache.")

print(f"\n1. Total rows (pages) in dataset: {len(df):,}")
print(f"2. Number of columns (features): {df.shape[1]}")
print("\nAvailable columns:")
print(list(df.columns))

# Inspect summary stats for numeric columns
display(df.describe().T[["count", "mean", "50%", "max"]].head(10))

Dataset loaded from local cache.

1. Total rows (pages) in dataset: 30,000
2. Number of columns (features): 44

Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,count,mean,50%,max
search_volume,27532.0,158.882391,10.0,74000.00
competition,27532.0,0.146954,0.0,1.00
cpc,27532.0,0.485342,0.0,100.36
word_count,22301.0,3107.760325,2877.0,9546.00
char_count,22301.0,20665.277835,19116.0,111158.00
impressions_90d,30000.0,5200.366300,731.0,517715.00
clicks_90d,30000.0,16.097333,1.0,4178.00
pageviews_90d,30000.0,49.942467,8.0,5998.00
sessions_90d,30000.0,37.066633,7.0,4345.00
users_90d,30000.0,35.937700,7.0,4913.00


In [9]:
print(f"3. Unique clients: {df['client_id'].nunique():,}")
print(f"4. Median days since last update: {df['days_since_last_update'].median():.0f} days")
print("\n5. Trend direction breakdown (% of pages):")
print((df["trend_direction"].value_counts(normalize=True) * 100).round(1))

3. Unique clients: 32
4. Median days since last update: 20 days

5. Trend direction breakdown (% of pages):
trend_direction
down      54.2
stable    19.9
up        14.6
new        7.5
flat       3.8
Name: proportion, dtype: float64


### Why These Numbers Make Content Refresh Worth 7 Weeks

* **Scale & Portfolio Breadth:** The starter dataset contains **30,000 pages** across **[N] clients** with **44 features** (combining 90-day and 30-day rolling search metrics, content age, freshness tiers, and engagement signals). Manually reviewing 30,000 pages is impossible for an editorial team.
* **Skewed Traffic Distribution:** While mean 90-day impressions are **5,200** (and up to **517,715** at the top), the median page gets **731 impressions** and only **1 click** in 90 days—meaning editorial effort must be carefully targeted rather than spread evenly.
* **Aging & Declining Content:** **[X%]** of pages in the dataset are classified as declining (`trend_direction`), and the median page has gone **[Y] days** since its last update. Identifying which decaying, high-impression pages are most recoverable gives editors an immediate, high-leverage triage list.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

### What I Can and Cannot Claim

* **What I Can Claim (Observational & Decision-Support):**
  * I can measure **historical patterns** in this anonymized dataset (e.g., which combinations of content age, `days_since_last_update`, position shift, and `trend_pct` are associated with declining clicks or impressions).
  * I can produce a **ranked priority shortlist** that helps an editorial team focus limited review hours on pages with high visibility and observable downward trends.
* **What I Cannot Claim (Causal or Universal):**
  * **No causal proof:** Because this is observational log data rather than a randomized experiment, I cannot claim that updating a specific page *caused* a ranking or traffic recovery, nor that `days_since_last_update` alone *causes* decay.
  * **No "predicting Google's algorithm":** Search rankings depend on external competitors, query intent shifts, and search engine updates not visible in these 44 features.
  * **Proxy labels & missingness:** Columns like `trend_direction` and `trend_pct` are historical period-over-period comparisons (`last_30d` vs. `prev_30d`), which can be noisy for seasonal topics or low-click pages, and fields such as `word_count` (~22,301 non-null out of 30,000) have missing values that must be handled carefully.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

### Self-Check

- [x] **Lane chosen:** Picked a provisional lane (**Content Refresh**) backed by the dataset structure and editorial workflow.
- [x] **Decision & action named:** Defined the unit of analysis (one page), the output (a ranked editorial triage list), the decision (which pages to refresh first), and the cost of false positives vs. false negatives.
- [x] **Real starter numbers included:** Loaded the 30,000-row, 44-column starter dataset in a code cell and cited real summary statistics (`impressions_90d`, `clicks_90d`, `days_since_last_update`, and `trend_direction`).
- [x] **More than "train a model":** Framed around editorial triage, cost asymmetry, and domain baselines rather than raw model training alone.
- [x] **Careful language & privacy:** Avoided causal claims, noted proxy/missing-data limitations, and used only anonymized IDs (`content_id`, `client_id`) with zero real client names, URLs, or private queries.